# DuckDB Data Studio Connection Starter
This notebook provides starter code to connect to the various interfaces exposed by DuckDB Data Studio, including:
1. **REST API Interface** (FastAPI running on port `8085` internally)
2. **Postgres Wire Protocol (Buena Vista)** (running on port `5433` internally, mapped to `5436` on the host)
3. **Direct DuckDB Connection** (Read-Only to avoid file locks)

This notebook is configured to query the pre-seeded `product_inventory` starter database.

--- 
## 1. Connecting to the FastAPI REST API
You can request catalog endpoints, query metered telemetry, or fetch data directly from any user-defined API endpoint.

In [1]:
import requests
import pandas as pd

# Inside the docker network, the hostname is 'duckdb-data-studio'
# If running outside Docker on the host, use 'http://localhost:8086'
BASE_URL = "http://duckdb-data-studio:8085"

print("Checking active API endpoints...")
try:
    response = requests.get(f"{BASE_URL}/api/list-endpoints")
    if response.status_code == 200:
        endpoints = response.json()
        print(f"Exposed endpoints: {len(endpoints)}")
        for ep in endpoints:
            print(f"- GET /api/{ep['path']} ({ep['description']})")
    else:
        print(f"Failed to load endpoints: {response.status_code}")
except Exception as e:
    print(f"Could not connect to FastAPI server: {e}")

Checking active API endpoints...
Exposed endpoints: 3
- GET /api/settings (Returns a list of DuckDB settings matching a prefix or pattern.)
- GET /api/test2 (Dummy endpoint for testing parameters)
- GET /api/products (Returns a filtered list of product inventory matching category, price, and stock parameters.)


### Fetching data from the starter `/api/products` endpoint with query parameters:
Filters the inventory by `category`, `min_price`, `max_price`, `min_stock`, and supports standard pagination (`limit`, `offset`).

In [2]:
# Request the pre-seeded products endpoint with filters
api_path = "products"
params = {
    "category": "Electronics",
    "min_price": 50.0,
    "min_stock": 10,
    "limit": 5
}

print(f"Fetching filtered data from: /api/{api_path} with parameters: {params}")
try:
    res = requests.get(f"{BASE_URL}/api/{api_path}", params=params)
    if res.status_code == 200:
        data = res.json()
        print(f"Returned count: {data['meta']['count']} | Has more: {data['meta']['has_more']}")
        df = pd.DataFrame(data['results'])
        display(df)
    else:
        print(f"Error fetching data: {res.status_code} - {res.text}")
except Exception as e:
    print(f"Request failed: {e}")

Fetching filtered data from: /api/products with parameters: {'category': 'Electronics', 'min_price': 50.0, 'min_stock': 10, 'limit': 5}
Returned count: 5 | Has more: True


,product_id,name,category,price,stock
0,1,Smartphone Nexus,Electronics,492.19,325
1,2,Aura Wireless Headphones,Electronics,367.70,220
2,3,Quantum Smartwatch,Electronics,324.52,186
3,4,Pulse ANC Earbuds,Electronics,432.16,334
4,5,Apex USB-C Dock,Electronics,178.92,233


--- 
## 2. Connecting to Postgres Wire Protocol (Buena Vista PGWire)
Buena Vista runs Postgres wire compatibility on top of DuckDB. You can query DuckDB tables using standard Postgres client packages like `psycopg2` or `sqlalchemy`.

In [3]:
# Note: Ensure psycopg2-binary is installed in your kernel
# !pip install psycopg2-binary pandas

import psycopg2
import pandas as pd
import warnings

# Suppress Pandas DBAPI2 connection warning since Buena Vista works best with raw psycopg2 connections
warnings.filterwarnings('ignore', message='.*pandas only supports SQLAlchemy.*')

# Connection parameters:
# Host: duckdb-data-studio (or localhost if executing outside the container stack)
# Port: 5433 (or 5436 if executing outside the container stack)
conn_string = "postgresql://postgres@duckdb-data-studio:5433/starter"

try:
    conn = psycopg2.connect(conn_string)
    print("Successfully connected to Buena Vista PGWire!\n")
    
    # Query product inventory table
    query = """
    SELECT category, COUNT(*) as product_count, AVG(price) as avg_price 
    FROM product_inventory 
    GROUP BY category 
    ORDER BY avg_price DESC;
    """
    df_stats = pd.read_sql(query, conn)
    display(df_stats)
    
    conn.close()
except Exception as e:
    print(f"Buena Vista PGWire connection failed: {e}")

Successfully connected to Buena Vista PGWire!



,category,product_count,avg_price
0,Electronics,6,347.648333
1,Apparel,6,315.861667
2,Beauty,6,270.016667
3,Books,6,231.070000
4,Home & Kitchen,6,207.041667
5,Sports,6,190.900000


--- 
## 3. Direct Remote/Local Connection via DuckDB Quack Protocol
DuckDB's native **Quack** extension (available in v1.5.3+) provides a high-performance HTTP client-server connection to remote DuckDB instances. Since Quack is currently in beta/experimental status, the standard `ATTACH` command can encounter catalog binding limitations. 

The recommended, fully supported approach to query remote tables is utilizing the **`quack_query`** function, or wrapping it in a **local SQL View** for a seamless, local-table-like experience without file locks.

In [4]:
import duckdb
import pandas as pd

# Connect to a local in-memory client session
con = duckdb.connect()

try:
    # Load the quack extension from the core_nightly repository
    con.execute("INSTALL quack FROM core_nightly;")
    con.execute("LOAD quack;")
    
    # Connection parameters
    host_uri = "quack:duckdb-studio:8001"
    token = "duckdb_studio_secret_token_123"
    
    print("1. Querying live data directly via quack_query...")
    query_sql = """
        SELECT * FROM quack_query(
            ?, 
            'SELECT category, COUNT(*) as count, AVG(price) as avg_price FROM product_inventory GROUP BY category;', 
            token=?,
            disable_ssl=true
        );
    """
    df_stats = con.execute(query_sql, [host_uri, token]).df()
    display(df_stats)
    
    print("\n2. Creating a local SQL View wrapping the remote Quack connection...")
    # This maps the remote 'product_inventory' table to a local view identifier
    con.execute(f"""
        CREATE OR REPLACE VIEW product_inventory AS 
        SELECT * FROM quack_query('{host_uri}', 'SELECT * FROM product_inventory;', token='{token}', disable_ssl=true);
    """)
    
    # Now you can query it natively as if it were a local table!
    df_view = con.execute("SELECT * FROM product_inventory WHERE category = 'Electronics' LIMIT 5;").df()
    display(df_view)
    
    con.close()
except Exception as e:
    print(f"Quack remote connection failed: {e}")


1. Querying live data directly via quack_query...


,category,count,avg_price
0,Electronics,6,347.648333
1,Books,6,231.070000
2,Sports,6,190.900000
3,Apparel,6,315.861667
4,Beauty,6,270.016667
5,Home & Kitchen,6,207.041667



2. Creating a local SQL View wrapping the remote Quack connection...


,product_id,name,category,price,stock
0,1,Smartphone Nexus,Electronics,492.19,325
1,2,Aura Wireless Headphones,Electronics,367.70,220
2,3,Quantum Smartwatch,Electronics,324.52,186
3,4,Pulse ANC Earbuds,Electronics,432.16,334
4,5,Apex USB-C Dock,Electronics,178.92,233
